# From GCSE quality to A-level: a joint model

The GCSE notebooks built three school-level quantities: a **general quality** $g_i$ (with regional means), a **tilt** $h_i$ (Maths and Science against English and Open) and a **consistency** $s_i$. This notebook asks how they carry into A-level value added, using a short list of subject groups.

| Group | A-level subjects pooled |
| --- | --- |
| Maths | Mathematics (Further Maths is left out: its pupils also take Maths, so they would be counted twice) |
| Sciences | Biology, Chemistry, Physics |
| English | English Literature, English Language, English Language & Literature |
| Humanities | History, Geography, Religious Studies, Philosophy, Ancient History, Classical Civilisation |
| Social sciences | Psychology, Sociology, Economics, Government & Politics, Law |
| Business & Computing | Business Studies, Computing |
| Creative arts | Art & Design (all types), Music, Music Technology, Drama & Theatre Studies, Dance |

A group's value added in a school is the mean of its subjects' value added, weighted by entries, with standard error from the published confidence intervals, treating the subjects' cohorts as separate. Where pupils take two subjects in a group (common in the sciences) that understates the true error somewhat; the school-level scatter term below absorbs part of it.

**Questions**

1. How much of each group's school-level A-level value added is explained by general GCSE quality, and by the tilt?
2. Is there a shared A-level quality that GCSE does not explain, and how big is it?
3. Do regions differ at A-level *given* their GCSE quality? (The humanities notebook found London above the line; here region is in both parts of the model.)
4. Is one A-level quality per school enough, related to GCSE quality as you originally proposed, or do the groups need their own slopes?

**Everything is fitted jointly.** The tilt is only moderately well determined for a single school (posterior SD about 0.63 against a prior of 1), so plugging in point estimates would shrink its A-level slopes by roughly a third to a half. Fitting the GCSE and A-level parts together carries that uncertainty through.

**This is association, not effect.** A school's GCSE value added is for its current Year 11, while its A-level students took GCSEs two years earlier and some joined from other schools. So the GCSE quantities are a proxy for the school, the slopes are weakened by that mismatch, and the A-level residual also absorbs sixth-form intake and subject selection.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt

%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")
print(f"Running on PyMC v{pm.__version__}")

## Data

The GCSE data are exactly as in the earlier notebooks (six elements per school with known standard errors, regions), and the A-level data are the seven pooled groups.

In [ ]:
raw = pd.read_csv("data/all-value-add-errors.csv")

elements = ["English", "Maths", "Science", "Humanities", "Languages", "Open"]
column = {"English": "P8MEAENG", "Maths": "P8MEAMAT", "Science": "SCIVAMEA_PTQ_EE",
          "Humanities": "HUMVAMEA_PTQ_EE", "Languages": "LANVAMEA_PTQ_EE", "Open": "P8MEAOPEN"}
frames = []
for e in elements:
    c = column[e]
    sub = raw[["URN", c, f"{c} lower", f"{c} upper"]].dropna()
    sub.columns = ["URN", "va", "lower", "upper"]
    sub["element"] = e
    frames.append(sub)
long = pd.concat(frames)
long["se"] = (long["upper"] - long["lower"]) / (2 * 1.96)
n_elements_per_school = long.groupby("URN")["element"].nunique()
long = long[long["URN"].isin(n_elements_per_school[n_elements_per_school >= 3].index)].sort_values("URN").reset_index(drop=True)

urns = pd.Index(sorted(long["URN"].unique()))
long["school_idx"] = urns.get_indexer(long["URN"])
long["element_idx"] = long["element"].map({e: k for k, e in enumerate(elements)}).to_numpy()
n_schools, n_elements = len(urns), len(elements)
x_obs, x_se = long["va"].to_numpy(), long["se"].to_numpy()
s_idx, e_idx = long["school_idx"].to_numpy(), long["element_idx"].to_numpy()
counts = np.bincount(s_idx, minlength=n_schools)

region_series = raw.drop_duplicates("URN").set_index("URN")["RGN24NM"].reindex(urns)
regions = list(region_series.value_counts().index)
reg_idx = region_series.map({r: k for k, r in enumerate(regions)}).fillna(len(regions)).astype(int).to_numpy()   # unknown region -> last index (national average)
print(f"{n_schools} schools, {len(long)} GCSE school-element observations; {(reg_idx == len(regions)).sum()} schools without a region")

In [ ]:
arts = ["Art & Design", "Art & Design (Fine Art)", "Art & Design (Photography)", "Art & Design (Graphics)", "Art & Design (Textiles)",
        "Art & Design (3d Studies)", "Art & Design (Critical Studies)", "Music", "Music Technology", "Drama & Theatre Studies", "Dance"]
groups = {"Maths": ["Mathematics"],
          "Sciences": ["Biology", "Chemistry", "Physics"],
          "English": ["English Literature", "English Language", "English Language & Literature"],
          "Humanities": ["History", "Geography", "Religious Studies", "Logic/ Philosophy", "Ancient History", "Classical Civilisation"],
          "Social sciences": ["Psychology", "Sociology", "Economics", "Government & Politics", "Law"],
          "Business & Computing": ["Business Studies:Single", "Computer Studies/Computing"],
          "Creative arts": arts}
group_names = list(groups)
n_groups = len(group_names)

raw_g = raw.set_index("URN").reindex(urns)
frames = []
for gname, subjects in groups.items():
    va = pd.DataFrame({s: raw_g[f"A-level {s} VA"] for s in subjects})
    se = pd.DataFrame({s: (raw_g[f"A-level {s} VA upper"] - raw_g[f"A-level {s} VA lower"]) / (2 * 1.96) for s in subjects})
    ent = pd.DataFrame({s: raw_g[f"A-level {s} entries"].where(va[s].notna(), 0).fillna(0) for s in subjects})
    total = ent.sum(axis=1)
    pooled_va = (va.fillna(0) * ent).sum(axis=1) / total.replace(0, np.nan)
    pooled_se = np.sqrt(((se.fillna(0) * ent) ** 2).sum(axis=1)) / total.replace(0, np.nan)   # entry-weighted; assumes separate cohorts
    f = pd.DataFrame({"school_idx": np.arange(n_schools), "group": gname, "va": pooled_va.to_numpy(), "se": pooled_se.to_numpy(), "entries": total.to_numpy()})
    frames.append(f.dropna(subset=["va", "se"]))
alevel = pd.concat(frames).reset_index(drop=True)
alevel["group_idx"] = alevel["group"].map({g: k for k, g in enumerate(group_names)}).to_numpy()
y_obs, y_se = alevel["va"].to_numpy(), alevel["se"].to_numpy()
ys_idx, yg_idx = alevel["school_idx"].to_numpy(), alevel["group_idx"].to_numpy()
groups_per_school = np.bincount(ys_idx, minlength=n_schools)
print(f"{len(alevel)} school-group A-level observations; groups per school: {({int(k): int(v) for k, v in zip(*np.unique(groups_per_school, return_counts=True))})}")
assert (y_se > 0).all()

### First look

Left: for each group, how many schools, how large the cohorts and standard errors are, and how much of the spread across schools is real (the last column removes the average sampling variance). Right: the raw correlation between each A-level group and each GCSE element, and between the A-level groups themselves, before any modelling.

In [ ]:
rows = []
for k, gname in enumerate(group_names):
    d = alevel[alevel["group_idx"] == k]
    rows.append({"group": gname, "schools": len(d), "median entries": d["entries"].median(), "median SE": d["se"].median(),
                 "mean VA": d["va"].mean(), "SD across schools": d["va"].std(),
                 "SD after removing noise": np.sqrt(max(d["va"].var() - (d["se"]**2).mean(), 0))})
display(pd.DataFrame(rows).set_index("group").round(3))

wide = alevel.pivot(index="school_idx", columns="group", values="va")[group_names]
gcse_wide = long.pivot(index="school_idx", columns="element", values="va")[elements]
cor_ag = pd.concat([wide, gcse_wide], axis=1).corr(min_periods=100).loc[group_names, elements]
cor_aa = wide.corr(min_periods=100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, mat, cols, title in [(axes[0], cor_ag, elements, "A-level group vs GCSE element"), (axes[1], cor_aa, group_names, "A-level group vs A-level group")]:
    ax.imshow(mat.to_numpy(), vmin=0, vmax=0.6, cmap="Blues")
    ax.set_xticks(range(len(cols)), cols, rotation=35, ha="right"); ax.set_yticks(range(len(group_names)), group_names)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f"{mat.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(f"Raw correlation: {title}")
    ax.grid(False)
plt.tight_layout()
plt.show()

## Models

**GCSE side** (as in the tilt notebook, with regional means of general quality):

$$
x_{ie} = \mu_e + \lambda_e\, g_i + \kappa_e\, h_i + \delta_{ie}, \quad \delta_{ie} \sim \text{Normal}(0, \tau_e s_i), \quad g_i \sim \text{Normal}(m_{r(i)}, 1), \quad h_i \sim \text{Normal}(0, 1)
$$

with $\log s_i = \sigma_s(\rho\,(g_i - m_r) + \sqrt{1-\rho^2}\, w_i)$, $\kappa_{Humanities} = 0$ (the anchor), and each observation measured with its known standard error.

**A-level side, two versions.** Let $y_{ij}$ be the value added of group $j$ in school $i$, observed with known standard error $\sigma_{ij}$, so $y^{obs}_{ij} \sim \text{Normal}(y_{ij}, \sigma_{ij})$.

*Flexible.* Every group has its own slopes on the two GCSE dimensions, plus a shared A-level quality that GCSE does not explain and a group-specific scatter:
$$
y_{ij} = \nu_j + b_j\, g_i + c_j\, h_i + \lambda^A_j\, (\psi_{r(i)} + u_i) + e_{ij}, \qquad u_i \sim \text{Normal}(0, 1), \quad e_{ij} \sim \text{Normal}(0, \sigma_j)
$$
Here $b_j$ and $c_j$ are the change in group $j$'s value added for one SD of general quality and of tilt, $u_i$ is the school's A-level quality beyond GCSE (shared across groups, loading $\lambda^A_j > 0$), $\psi_r$ is a regional shift in that beyond-GCSE quality (zero-sum across regions, spread $\sigma_\psi$), and $e_{ij}$ is scatter specific to the group. As with $\delta_{ie}$, the group scatter is folded into the observation variance, $\sigma_j^2 + \sigma_{ij}^2$.

*One A-level quality.* Your original proposal: a single latent A-level quality per school, tied to the GCSE quantities, on which every group loads:
$$
a_i = \beta_g\, g_i + \beta_h\, h_i + \psi_{r(i)} + \sigma_u\, u_i, \qquad y_{ij} = \nu_j + \lambda^A_j\, a_i + e_{ij}
$$
with $\lambda^A_{Maths} = 1$ fixing the scale. This is the flexible model with the constraint $b_j = \lambda^A_j \beta_g$ and $c_j = \lambda^A_j \beta_h$: every group responds to the GCSE dimensions in the same proportion as it loads on the shared quality. If the data agree, the constrained model describes them as well and is simpler; if not, the groups differ in how GCSE reaches them.

We fit both and compare what they say.

**The sign of the tilt is arbitrary.** Flipping $h_i$, $\kappa_e$ and the A-level slopes on the tilt together gives exactly the same fit, so chains can settle on either version. After sampling we flip each chain to the orientation in which Maths and Science load positively and English and Open negatively, as in the tilt notebook. (An earlier version forced Maths to load positively inside the model; that pushed the mirror-image solution onto a boundary and one chain landed there, with a different solution and $\hat R$ up to 1.5.)

In [ ]:
is_maths = np.array([e == "Maths" for e in elements])
keep = np.array([0.0 if e == "Humanities" else 1.0 for e in elements])
is_maths_group = np.array([g == "Maths" for g in group_names])

def build_model(kind):
    """kind = 'flex': each A-level group has its own slopes on general quality g and tilt h.
       kind = 'latent': one A-level quality a_i = beta_g g_i + beta_h h_i + region + u_i, and groups load on it."""
    with pm.Model(coords={"element": elements, "region": regions, "group": group_names}) as model:
        # ---- GCSE side: general quality (regional means), tilt, consistency
        mu = pm.Normal("mu", 0, 1, dims="element")
        lam = pm.HalfNormal("lam", 1, dims="element")
        tau = pm.HalfNormal("tau", 0.5, dims="element")
        sigma_m = pm.HalfNormal("sigma_m", 0.5)
        m = pm.ZeroSumNormal("m", sigma=sigma_m, dims="region")
        m_all = pt.concatenate([m, pt.zeros(1)])
        g = pm.Normal("g", m_all[reg_idx], 1, shape=n_schools)
        h = pm.Normal("h", 0, 1, shape=n_schools)
        k_raw = pm.Normal("kappa_raw", 0, 0.5, shape=n_elements)
        kappa = pm.Deterministic("kappa", k_raw * keep, dims="element")   # Humanities fixed at 0; the sign of the tilt is fixed after sampling
        sigma_s = pm.HalfNormal("sigma_s", 0.5)
        rho = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
        w = pm.Normal("w", 0, 1, shape=n_schools)
        log_s = pm.Deterministic("log_s", sigma_s * (rho * (g - m_all[reg_idx]) + pt.sqrt(1 - rho**2) * w))
        pm.Normal("x_obs", mu=mu[e_idx] + lam[e_idx] * g[s_idx] + kappa[e_idx] * h[s_idx],
                  sigma=pt.sqrt((tau[e_idx] * pt.exp(log_s[s_idx]))**2 + x_se**2), observed=x_obs)

        # ---- A-level side
        nu = pm.Normal("nu", 0, 0.5, dims="group")
        sd_group = pm.HalfNormal("sd_group", 0.5, dims="group")           # group-specific school-level scatter
        sigma_psi = pm.HalfNormal("sigma_psi", 0.3)
        psi = pm.ZeroSumNormal("psi", sigma=sigma_psi, dims="region")      # regional shift in A-level quality beyond GCSE
        psi_all = pt.concatenate([psi, pt.zeros(1)])
        u = pm.Normal("u", 0, 1, shape=n_schools)                          # school's shared A-level quality
        if kind == "flex":
            b = pm.Normal("b", 0, 0.5, dims="group")                       # slope on general quality g
            c = pm.Normal("c", 0, 0.5, dims="group")                       # slope on tilt h
            lam_a_raw = pm.HalfNormal("lam_a", 0.5, dims="group")
            shared = psi_all[reg_idx] + u
            y_mean = nu[yg_idx] + b[yg_idx] * g[ys_idx] + c[yg_idx] * h[ys_idx] + lam_a_raw[yg_idx] * shared[ys_idx]
        else:
            beta_g = pm.Normal("beta_g", 0, 0.5)
            beta_h = pm.Normal("beta_h", 0, 0.5)
            sigma_u = pm.HalfNormal("sigma_u", 0.5)
            lam_raw = pm.HalfNormal("lam_raw", 1, shape=n_groups)
            lam_a = pm.Deterministic("lam_a", pt.where(is_maths_group, 1.0, lam_raw), dims="group")   # Maths sets the scale of A-level quality
            quality = beta_g * g + beta_h * h + psi_all[reg_idx] + sigma_u * u
            y_mean = nu[yg_idx] + lam_a[yg_idx] * quality[ys_idx]
        pm.Normal("y_obs", mu=y_mean, sigma=pt.sqrt(sd_group[yg_idx]**2 + y_se**2), observed=y_obs)
    return model

In [ ]:
flex_model = build_model("flex")
latent_model = build_model("latent")

### Fit

In [ ]:
with flex_model:
    idata_flex = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)

In [ ]:
with latent_model:
    idata_latent = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)

In [ ]:
import xarray as xr

def align_sign(idata, tilt_terms):
    """Flip each chain to the orientation with Maths and Science loading positively and English and Open negatively on the tilt."""
    post = idata.posterior
    k = post["kappa"]
    score = (k.sel(element="Maths") + k.sel(element="Science") - k.sel(element="English") - k.sel(element="Open")).mean("draw")
    sign = xr.where(score > 0, 1.0, -1.0)          # one sign per chain
    out = post.copy()
    for name in tilt_terms:
        out[name] = post[name] * sign
    return out, sign.to_numpy()

post_f, sign_f = align_sign(idata_flex, ["h", "kappa", "c"])
post_l, sign_l = align_sign(idata_latent, ["h", "kappa", "beta_h"])
print("chains flipped: flexible", int((sign_f < 0).sum()), "of", len(sign_f), "| one A-level quality", int((sign_l < 0).sum()), "of", len(sign_l))

### Diagnostics

In [ ]:
skip = {"kappa_raw", "rho_raw", "lam_raw"}    # raw parameters that enter only through transformed versions (checked via kappa, rho, lam_a)
for name, idata, post_ in [("flexible", idata_flex, post_f), ("one A-level quality", idata_latent, post_l)]:
    rh, es = az.rhat(post_), az.ess(post_)
    worst = pd.DataFrame({"max r_hat": {v: float(rh[v].max()) for v in rh.data_vars if v not in skip},
                          "min bulk ESS": {v: float(es[v].min()) for v in es.data_vars if v not in skip}}).sort_values("max r_hat", ascending=False)
    print(f"{name}: divergences = {int(idata.sample_stats['diverging'].sum())}")
    display(worst.round(3).head(8))

## How much of the A-level school effect does GCSE explain?

For each group, the true between-school variance of A-level value added splits into: **general GCSE quality** ($b_j^2$ times the variance of $g_i$), **tilt** ($c_j^2$ times the variance of $h_i$), the **regional** part of the beyond-GCSE quality, the **shared A-level residual** $u_i$, and the **group's own scatter** $\sigma_j^2$. The shares below are computed draw by draw from the flexible model.

In [ ]:
def flat(post, name): return post[name].to_numpy().reshape(-1, *post[name].shape[2:])

g_f, h_f, u_f = flat(post_f, "g"), flat(post_f, "h"), flat(post_f, "u")
psi_f = np.concatenate([flat(post_f, "psi"), np.zeros((g_f.shape[0], 1))], axis=1)
b_f, c_f, lamA_f, sdg_f = flat(post_f, "b"), flat(post_f, "c"), flat(post_f, "lam_a"), flat(post_f, "sd_group")
V_g, V_h, V_u = g_f.var(axis=1), h_f.var(axis=1), u_f.var(axis=1)
V_reg = psi_f[:, reg_idx].var(axis=1)

comp = {"general GCSE quality": b_f**2 * V_g[:, None], "tilt": c_f**2 * V_h[:, None],
        "regional (beyond GCSE)": lamA_f**2 * V_reg[:, None], "shared A-level residual": lamA_f**2 * V_u[:, None], "group's own scatter": sdg_f**2}
total = sum(comp.values())
shares = {k: (v / total) for k, v in comp.items()}
tab = pd.DataFrame({k: v.mean(axis=0) for k, v in shares.items()}, index=group_names)
tab["true SD"] = np.sqrt(total).mean(axis=0)
tab["GCSE explains (g + tilt)"] = (shares["general GCSE quality"] + shares["tilt"]).mean(axis=0)
lo, hi = np.percentile(shares["general GCSE quality"] + shares["tilt"], [5.5, 94.5], axis=0)
tab["89% interval"] = [f"[{a:.2f}, {b:.2f}]" for a, b in zip(lo, hi)]
display(tab.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
left = np.zeros(n_groups)
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B3", "#CCB974"]
for (k, _), col in zip(comp.items(), colors):
    v = shares[k].mean(axis=0)
    ax.barh(group_names, v, left=left, color=col, label=k)
    left += v
ax.invert_yaxis(); ax.set_xlabel("share of true between-school variance"); ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=8)
plt.show()

### Which GCSE dimension reaches which group?

The slopes on general quality ($b_j$) and on the tilt ($c_j$), per SD of the GCSE quantity, in A-level value-added points. If the tilt is a real Maths-and-Science versus English-and-Open dimension, $c_j$ should be positive for Maths and Sciences and negative for English (and perhaps Creative arts, if it reflects the Open side).

In [ ]:
def forest(ax, draws, label, color="#4C72B0", offset=0.0, name=None):
    lo, med, hi = np.percentile(draws, [5.5, 50, 94.5], axis=0)
    for k in range(draws.shape[1]):
        ax.plot([lo[k], hi[k]], [k + offset, k + offset], color=color, linewidth=2, label=name if k == 0 else None)
        ax.plot(med[k], k + offset, "o", color=color)
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_xlabel(label)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
forest(axes[0], b_f, r"slope on general GCSE quality, $b_j$")
forest(axes[1], c_f, r"slope on tilt (+: Maths/Science side), $c_j$", color="#DD8452")
axes[0].set_yticks(range(n_groups), group_names); axes[0].invert_yaxis()
plt.tight_layout(); plt.show()
pd.DataFrame({"b (general quality)": b_f.mean(axis=0), "P(b > 0)": (b_f > 0).mean(axis=0),
              "c (tilt)": c_f.mean(axis=0), "P(c > 0)": (c_f > 0).mean(axis=0), "P(c < 0)": (c_f < 0).mean(axis=0),
              "shared-quality loading": lamA_f.mean(axis=0), "group scatter": sdg_f.mean(axis=0)}, index=group_names).round(3)

## Do regions differ at A-level, given GCSE?

Left: the regional shift in general GCSE quality, $m_r$. Right: the regional shift in the A-level quality that GCSE does not explain, $\psi_r$, from the flexible model. A region can be ahead at GCSE, at A-level beyond GCSE, both or neither; the second is what the earlier subject notebooks could not separate.

In [ ]:
m_f, psi_only = flat(post_f, "m"), flat(post_f, "psi")
order_r = np.argsort(-m_f.mean(axis=0))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
for ax, d, label, col in [(axes[0], m_f, r"GCSE general quality, $m_r$", "#4C72B0"), (axes[1], psi_only, r"A-level shift beyond GCSE, $\psi_r$", "#DD8452")]:
    lo, med, hi = np.percentile(d, [5.5, 50, 94.5], axis=0)
    for row, k in enumerate(order_r):
        ax.plot([lo[k], hi[k]], [row, row], color=col, linewidth=2); ax.plot(med[k], row, "o", color=col)
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--"); ax.set_xlabel(label)
axes[0].set_yticks(range(len(regions)), [regions[k] for k in order_r]); axes[0].invert_yaxis()
plt.tight_layout(); plt.show()
print("sigma_m (GCSE regional spread):", f"{flat(post_f, 'sigma_m').mean():.3f}", "| sigma_psi (A-level regional spread beyond GCSE):", f"{flat(post_f, 'sigma_psi').mean():.3f} [{np.percentile(flat(post_f, 'sigma_psi'), 5.5):.3f}, {np.percentile(flat(post_f, 'sigma_psi'), 94.5):.3f}]")
pd.DataFrame({"m_r (GCSE)": m_f.mean(axis=0), "psi_r (A-level)": psi_only.mean(axis=0), "P(psi_r > 0)": (psi_only > 0).mean(axis=0)}, index=regions).iloc[order_r].round(3)

## Is one A-level quality enough?

The one-quality model says every group responds to the GCSE dimensions in proportion to how it loads on the shared quality. We compare, for each group, the flexible slopes with what that constraint implies, $\lambda^A_j \beta_g$ and $\lambda^A_j \beta_h$. Then the same residual check as in the GCSE notebooks: the correlation across schools of each model's standardised A-level residuals, for schools with all seven groups, against a replicated null. If the shared quality captured the cross-group structure, these should be near zero.

In [ ]:
beta_g, beta_h, sigma_u = flat(post_l, "beta_g"), flat(post_l, "beta_h"), flat(post_l, "sigma_u")
lamA_l = flat(post_l, "lam_a")
imp_b, imp_c = lamA_l * beta_g[:, None], lamA_l * beta_h[:, None]
print("one-quality model: beta_g =", f"{beta_g.mean():.3f} [{np.percentile(beta_g, 5.5):.3f}, {np.percentile(beta_g, 94.5):.3f}]",
      "| beta_h =", f"{beta_h.mean():.3f} [{np.percentile(beta_h, 5.5):.3f}, {np.percentile(beta_h, 94.5):.3f}]",
      "| sigma_u =", f"{sigma_u.mean():.3f}")
g_l, h_l, u_l = flat(post_l, "g"), flat(post_l, "h"), flat(post_l, "u")
psi_l = np.concatenate([flat(post_l, "psi"), np.zeros((g_l.shape[0], 1))], axis=1)
a_l = beta_g[:, None] * g_l + beta_h[:, None] * h_l + psi_l[:, reg_idx] + sigma_u[:, None] * u_l
V_a = a_l.var(axis=1)
corr_ag = np.array([np.corrcoef(a_l[d], g_l[d])[0, 1] for d in range(len(a_l))])
corr_ah = np.array([np.corrcoef(a_l[d], h_l[d])[0, 1] for d in range(len(a_l))])
expl = ((beta_g[:, None] * g_l).var(axis=1) + (beta_h[:, None] * h_l).var(axis=1)) / V_a
print("correlation of A-level quality with general GCSE quality:", f"{corr_ag.mean():.3f} [{np.percentile(corr_ag, 5.5):.3f}, {np.percentile(corr_ag, 94.5):.3f}]",
      "| with the tilt:", f"{corr_ah.mean():.3f}", "| share of A-level quality variance explained by g and tilt:", f"{expl.mean():.3f} [{np.percentile(expl, 5.5):.3f}, {np.percentile(expl, 94.5):.3f}]")
pd.DataFrame({"flexible b": b_f.mean(axis=0), "implied b (loading x beta_g)": imp_b.mean(axis=0), "flexible c": c_f.mean(axis=0), "implied c (loading x beta_h)": imp_c.mean(axis=0),
              "loading, flexible": lamA_f.mean(axis=0), "loading, one-quality": lamA_l.mean(axis=0)}, index=group_names).round(3)

In [ ]:
six = groups_per_school == n_groups
rows7 = six[ys_idx]
order7 = np.lexsort((yg_idx[rows7], ys_idx[rows7]))
thin = slice(0, 1000, 5)
def sub(post, name):
    a = post[name].to_numpy()[:, thin]; return a.reshape(-1, *a.shape[2:])

def a_resid_corr(post, kind):
    nu, sdg = sub(post, "nu"), sub(post, "sd_group")
    g, h, u = sub(post, "g"), sub(post, "h"), sub(post, "u")
    psi = np.concatenate([sub(post, "psi"), np.zeros((len(nu), 1))], axis=1)[:, reg_idx]
    lamA = sub(post, "lam_a")
    if kind == "flex":
        mean = nu[:, yg_idx] + sub(post, "b")[:, yg_idx] * g[:, ys_idx] + sub(post, "c")[:, yg_idx] * h[:, ys_idx] + lamA[:, yg_idx] * (psi + u)[:, ys_idx]
    else:
        q = sub(post, "beta_g")[:, None] * g + sub(post, "beta_h")[:, None] * h + psi + sub(post, "sigma_u")[:, None] * u
        mean = nu[:, yg_idx] + lamA[:, yg_idx] * q[:, ys_idx]
    sd = np.sqrt(sdg[:, yg_idx]**2 + y_se**2)
    out = {}
    for label, r in [("observed", (y_obs[None, :] - mean) / sd), ("replicated", rng.standard_normal(mean.shape))]:
        r7 = r[:, rows7][:, order7].reshape(r.shape[0], -1, n_groups)
        out[label] = np.stack([np.corrcoef(r7[d].T) for d in range(r7.shape[0])])
    return out

res_f, res_l = a_resid_corr(post_f, "flex"), a_resid_corr(post_l, "latent")
fig, axes = plt.subplots(1, 3, figsize=(15, 5), gridspec_kw={"width_ratios": [1, 1, 0.04]})
for ax, res, title in [(axes[0], res_f, "flexible"), (axes[1], res_l, "one A-level quality")]:
    corr = res["observed"].mean(axis=0)
    im = ax.imshow(corr, vmin=-0.3, vmax=0.3, cmap="RdBu_r")
    ax.set_xticks(range(n_groups), group_names, rotation=35, ha="right"); ax.set_yticks(range(n_groups), group_names)
    for i in range(n_groups):
        for j in range(n_groups):
            ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(f"A-level residual correlation, {title}"); ax.grid(False)
fig.colorbar(im, cax=axes[2])
plt.show()
null_hi = np.percentile(res_f["replicated"][:, 0, 1], 94.5)
off = ~np.eye(n_groups, dtype=bool)
print(f"{six.sum()} schools with all {n_groups} groups; null (replicated) 89% upper bound for a single pair is about {null_hi:.3f}")
print("largest off-diagonal residual correlation: flexible", np.abs(res_f['observed'].mean(axis=0)[off]).max().round(3), "| one A-level quality", np.abs(res_l['observed'].mean(axis=0)[off]).max().round(3))

## Summary

Both models converge after the chains are aligned on the sign of the tilt (worst $\hat R$ 1.06 for the flexible model, 1.03 for the one-quality model; the weakest parameter is still the overall level $\mu_e$, as in the GCSE notebooks). All slopes are per SD of the GCSE quantity, in A-level value-added points, and describe association, not effect.

**1. GCSE explains a modest share, and it is a different share for different groups.** The share of each group's true between-school variance that general GCSE quality and the tilt together explain is 28% for Maths [22, 34], 28% for Sciences [22, 33], 19% for Humanities, 17% for English, 12% for Social sciences, 8% for Business & Computing and 4% for Creative arts. This is in line with the 5-15% found in the earlier subject notebooks, with the two quantitative groups higher.

- **General quality reaches every group about equally.** The slope $b_j$ is 0.08 to 0.10 for all seven groups: a school one SD higher in general GCSE quality has A-level value added about 0.1 points higher in every group. The shares differ because the groups' true spread differs (0.26 for Humanities against 0.44 for Creative arts), not because the slope does.
- **The tilt reaches Maths and Sciences strongly and the others hardly at all.** The slope $c_j$ on the tilt is $+0.18$ for Maths and $+0.15$ for Sciences (both certainly positive), against $-0.05$ for English, $-0.03$ for Humanities and $-0.04$ for Social sciences (small, but negative with probability above 0.99), and about zero for Business & Computing ($+0.02$) and Creative arts ($-0.03$). For Maths and Sciences the tilt explains about 20% of the true variance, *more than general quality does* (7-8%). A school that leans to Maths and Science at GCSE, relative to its general quality, leans the same way at A-level: this is the subject-matched persistence one would expect, and the largest single GCSE effect in the model.

**2. There is a shared A-level quality that GCSE does not explain, and it is the largest part for most groups.** It accounts for 45-55% of the variance for Maths, Sciences, Humanities, Social sciences and Business & Computing, but only 19% for English and 7% for Creative arts. The groups' own scatter takes the rest, most of all for Creative arts (89%) and English (64%). The own-scatter share also absorbs measurement error the standard errors do not capture (small cohorts, and pupils counted in two subjects of a pooled group), so it is an upper bound and the GCSE shares above are, if anything, understated.

**3. Regions: mostly carried by GCSE, with a few shifts beyond it.** The regional spread of the A-level quality beyond GCSE is $\sigma_\psi = 0.14$ [0.06, 0.25], small next to the loadings, so the regional part of the variance is under 1% in every group. In points for Maths (the largest loading, 0.27), London is about 0.06 above the national average through its GCSE quality and only about 0.02 more beyond it; the North East is about 0.035 below through GCSE and about 0.04 below beyond it; the West Midlands is close to the national average through GCSE but about 0.035 below beyond it; the East of England is about 0.045 above beyond GCSE (probability positive 0.99). Only the East of England and West Midlands shifts are reasonably clear.

**4. One A-level quality is not enough.** Your original proposal, a single A-level quality $a_i = \beta_g g_i + \beta_h h_i + \psi_r + \sigma_u u_i$, gives $\beta_g = 0.126$ [0.110, 0.142], $\beta_h = 0.057$ [0.037, 0.078] and a correlation of 0.42 [0.39, 0.44] between A-level quality and general GCSE quality (0.18 with the tilt); GCSE explains 20% of its variance. But the model cannot represent how GCSE reaches the groups: it forces the tilt slopes to be in proportion to the loadings (all positive, 0.02 to 0.06), where the data say $+0.18$ and $+0.15$ for Maths and Sciences and slightly negative for the others. The residual check confirms it: with one quality the residual correlations are $+0.23$ (Maths-Sciences) and $+0.19$ (Humanities-Social sciences), with a negative block between the Maths and Sciences pair and English, Humanities and Social sciences (down to $-0.12$) against a null of about $\pm 0.06$; the flexible model leaves at most 0.10. The one-quality version is still a useful summary of the overall link (0.42), but the groups need their own slopes.

**Creative arts and the Open side.** If the tilt's English-and-Open side reflected breadth, one might expect Creative arts to lean that way. It does, weakly: $c = -0.026$, negative with probability 0.87, but the effect is negligible against a group spread of 0.44, and GCSE explains only 4% of its variance. Creative arts A-level results are mostly their own thing here (and its cohorts are small).

**Caveats.** The GCSE quantities describe this year's Year 11, not the A-level students' cohort; pooled standard errors assume separate cohorts; one year of data. The tilt is defined relative to the Humanities anchor.

**Next.** (a) Add each school's specific GCSE strength in the matching element, $\delta_{ie}$, to test whether it adds beyond general quality and the tilt; (b) link A-level scatter across groups to GCSE consistency $s_i$; (c) use the Maths-Sciences result to check whether it survives replacing the pooled Sciences group by the three subjects.